In [ ]:
pip install dotenv

In [ ]:
from tools import *

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [ ]:
groq_model_client = OpenAIChatCompletionClient(
    model="llama-3.1-8b-instant",  # 👈 change this
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "structured_output": False,
        "family": "unknown",
    }
)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("GROQ_API_KEY"))

In [ ]:
from autogen_agentchat.agents import AssistantAgent

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat

In [ ]:
from autogen_agentchat.conditions import ExternalTermination, TextMentionTermination

In [ ]:
from autogen_agentchat.ui import Console

In [ ]:
import asyncio

In [ ]:
from autogen_core.models import UserMessage
from autogen_ext.models.ollama import OllamaChatCompletionClient

In [ ]:
from tools import *

In [ ]:
import tools

print(dir(tools))

In [ ]:
import importlib
import tools

importlib.reload(tools)

from tools import *

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from tools import *

coding_agent = AssistantAgent(
    "coding_agent",
    model_client=groq_model_client,
    tools=[create_folder, create_file, read_file, edit_file, list_files],
    system_message="""
You are a senior software engineer.

Responsibilities:
- Build the project requested by the user.
- Create folders and files when needed.
- Write complete, working code.
- Fix all reviewer feedback.
- Continue until reviewer says APPROVE.

STRICT RULES:
- Always use relative paths only (e.g., todo_app/main.py). NEVER use /home/user/... or any absolute path.
- When creating a project in a folder, first create the folder, then create files ONE BY ONE separately. Do NOT create folder and files in the same parallel call.
- Keep file content simple. Use single quotes inside Python code to avoid escaping issues.
- Never review your own code.
"""
)

review_agent = AssistantAgent(
    "review_agent",
    model_client=groq_model_client,
    tools=[read_file, list_files],
    system_message="""
You are a strict senior code reviewer.

Responsibilities:
- Call list_files to see what files exist.
- Call read_file to read each file one by one.
- Check for bugs, missing features, bad practices.

STRICT RULES:
- You can ONLY use read_file and list_files. Nothing else.
- NEVER call write_file, create_file, edit_file or any other tool.
- Only use relative paths (e.g., todo_app/main.py). NEVER use absolute paths.
- If issues found: describe them clearly in plain text for the coding agent to fix.
- If everything looks correct: reply with exactly this word: APPROVE
"""
)

In [ ]:
from autogen_agentchat.conditions import TextMentionTermination

termination = TextMentionTermination("APPROVE")

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    participants=[coding_agent, review_agent],
    termination_condition=termination
)

In [ ]:
await Console(
    team.run_stream(
        task="Create fact.py which has factorial code"
    )
)

In [ ]:
await Console(
    team.run_stream(
        task="Create a todo app in a folder called todo_app. It should have add, view, delete tasks. Save tasks to a tasks.json file."
    )
)


In [ ]:
await Console(
    team.run_stream(
        task="Create a binary.py and write code of binary search in that"
    )
)
